<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/01_Data_Scientist/intermediate/03_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature Engineering

Feature engineering is often more impactful than model selection. This notebook covers professional techniques for transforming raw data into predictive features.

**Topics:** Encoding, scaling, date features, interaction features, feature selection, ColumnTransformer, handling imbalanced data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler,
                                    LabelEncoder, OneHotEncoder, OrdinalEncoder)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import SelectKBest, f_classif, RFE, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)

# Generate realistic dataset
n = 5000
df = pd.DataFrame({
    'age':         np.random.randint(18, 75, n),
    'income':      np.random.lognormal(10.5, 0.8, n),
    'credit_score':np.random.normal(680, 100, n).clip(300, 850),
    'tenure_days': np.random.exponential(500, n).astype(int),
    'employment':  np.random.choice(['employed', 'self_employed', 'unemployed', 'retired'], n, p=[0.6,0.2,0.1,0.1]),
    'education':   np.random.choice(['high_school', 'bachelor', 'master', 'phd'], n, p=[0.3,0.4,0.2,0.1]),
    'join_date':   pd.date_range('2018-01-01', periods=n, freq='1H'),
    'defaulted':   np.random.choice([0, 1], n, p=[0.8, 0.2]),
})
print('Shape:', df.shape, '| Target rate:', df['defaulted'].mean())

## 1. Categorical Encoding

In [ ]:
print('=== Categorical Encoding Methods ===')
sample_cats = pd.Series(['A', 'B', 'C', 'A', 'B', 'A'])

# 1. Label Encoding (ordinal-like, use for tree models only)
le = LabelEncoder()
label_encoded = le.fit_transform(sample_cats)
print('Label Encoding:', dict(zip(sample_cats.unique(), le.transform(sample_cats.unique()))))
print('  ⚠️  Implies ordinal relationship — only use with tree models!\n')

# 2. One-Hot Encoding (for nominal categories)
ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' avoids dummy variable trap
ohe_encoded = ohe.fit_transform(sample_cats.values.reshape(-1,1))
print('One-Hot Encoding (drop=first):')
print(pd.DataFrame(ohe_encoded, columns=['B', 'C']))
print('  ✓ Use for nominal categories with linear models\n')

# 3. Target Encoding (mean of target per category)
def target_encode(df, col, target, smoothing=10):
    """Smoothed target encoding (reduces overfitting on rare categories)."""
    global_mean = df[target].mean()
    agg = df.groupby(col)[target].agg(['mean', 'count'])
    smooth = (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)
    return df[col].map(smooth)

df['employment_te'] = target_encode(df, 'employment', 'defaulted')
print('Target Encoding (employment → default rate):')
print(df.groupby('employment')['defaulted'].mean().round(3))
print('Encoding values:')
print(df.groupby('employment')['employment_te'].first().round(3))

# 4. Ordinal Encoding (education has natural order)
education_order = ['high_school', 'bachelor', 'master', 'phd']
df['education_ordinal'] = df['education'].map({v: i for i, v in enumerate(education_order)})
print('\nOrdinal Encoding (education):', {v: i for i, v in enumerate(education_order)})

## 2. Numerical Scaling

In [ ]:
# Scaling is critical for: linear models, SVMs, KNN, neural networks
# Tree-based models (RF, XGBoost) don't need scaling

income_sample = df['income'].values.reshape(-1,1)

scalers = {
    'StandardScaler': StandardScaler(),      # z-score, sensitive to outliers
    'MinMaxScaler':   MinMaxScaler(),         # [0,1] range, sensitive to outliers
    'RobustScaler':   RobustScaler(),         # uses IQR — robust to outliers!
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].hist(income_sample, bins=50, color='steelblue', alpha=0.7)
axes[0].set_title('Original Income\n(right-skewed)')

for ax, (name, scaler) in zip(axes[1:], scalers.items()):
    scaled = scaler.fit_transform(income_sample)
    ax.hist(scaled, bins=50, color='darkorange', alpha=0.7)
    ax.set_title(f'{name}\nmean={scaled.mean():.2f} std={scaled.std():.2f}')

plt.suptitle('Effect of Different Scaling Methods on Skewed Data', fontsize=12)
plt.tight_layout(); plt.show()

# Log transform for right-skewed data
df['log_income'] = np.log1p(df['income'])  # log(1+x) handles zero
print('Log transform reduces skewness:')
print(f'  Income skewness:     {df["income"].skew():.2f}')
print(f'  log(income) skewness:{df["log_income"].skew():.2f}')

## 3. Feature Creation & Date Features

In [ ]:
# Date features — essential for time-based datasets
df['join_year']  = df['join_date'].dt.year
df['join_month'] = df['join_date'].dt.month
df['join_dow']   = df['join_date'].dt.dayofweek  # 0=Monday
df['join_quarter'] = df['join_date'].dt.quarter
df['is_weekend'] = (df['join_date'].dt.dayofweek >= 5).astype(int)

# Interaction features
df['age_income_ratio']    = df['age'] / (df['income'] + 1)
df['credit_income_ratio'] = df['credit_score'] / (df['income'] + 1) * 1000
df['income_per_tenure']   = df['income'] / (df['tenure_days'] + 1)

# Age bins
df['age_group'] = pd.cut(df['age'], bins=[0, 25, 35, 50, 100], 
                          labels=['<25', '25-35', '35-50', '50+'])

print('New features created:')
new_features = ['join_year', 'join_month', 'is_weekend', 'age_income_ratio', 
                'credit_income_ratio', 'age_group']
print(df[new_features].head())

## 4. sklearn ColumnTransformer — Professional Pipeline

In [ ]:
from sklearn.impute import SimpleImputer

# Define which transformation applies to which columns
numeric_features = ['age', 'log_income', 'credit_score', 'tenure_days',
                    'age_income_ratio', 'credit_income_ratio']
categorical_features = ['employment', 'education']
ordinal_features = ['education_ordinal']

# Pipeline for numeric columns: impute then scale
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
])

# Pipeline for categorical columns: impute then OHE
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# ColumnTransformer: apply the right transformer to the right columns
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
], remainder='drop')

# Full pipeline: preprocessing + model
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

X = df[numeric_features + categorical_features]
y = df['defaulted']

scores = cross_val_score(pipeline, X, y, cv=5, scoring='roc_auc')
print(f'Pipeline CV AUC: {scores.mean():.4f} ± {scores.std():.4f}')
print('\n✓ This entire preprocessing + model pipeline:')
print('  - Prevents data leakage (fit only on training folds)')
print('  - Can be saved and deployed as a single object')
print('  - Is reproducible and testable')